# Raw data 정리

In [3]:
%pip install pandas
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import os
import glob
import re

# 1. get raw data files
# demand data => monthly data
df_demand = pd.read_csv('../data/demand.csv')
df_demand.rename(columns={'year_month': 'date', '총계': 'demand'}, inplace=True)
df_demand = df_demand[['date', 'demand']]
df_demand['date'] = pd.to_datetime(df_demand['date'], format='%Y%m')

# weather data => daily data
df_weather = pd.read_csv('../data/korea_daily_avg_ta_rn_20010101_20250831.csv')
df_weather = df_weather[['date', 'ta_avg_mean','rn_day_mean']]

# get price data files and merge them => daily data
files = sorted(glob.glob(os.path.join("../data/price*.csv")))
dfs = []
for f in files:
    df = pd.read_csv(f)
    # 1) 날짜 컬럼만 골라내기 (예: '2001.1.2', '2010.12.21' 형태)
    date_pat = re.compile(r"^\d{4}\.\d{1,2}\.\d{1,2}$")
    date_cols = [c for c in df.columns if isinstance(c, str) and date_pat.match(c)]

    # (선택) 필요한 Quotation Source만 필터링 (두 개만 원할 때)
    need_sources = ["PLATT'S CRUDE", "PLATT'S AP/AG"]
    df = df[df["Quotation Source"].isin(need_sources)]

    # 2) wide → long
    long = df.melt(
        id_vars=["Quotation Source"],
        value_vars=date_cols,
        var_name="date",
        value_name="price"
    )

    # 3) 타입 정리
    long["date"] = pd.to_datetime(long["date"], format="%Y.%m.%d", errors="coerce")
    long["price"] = pd.to_numeric(long["price"], errors="coerce")

    # 4) long → wide (행=날짜, 열=Quotation Source)
    df_price_pivot = (
        long.pivot_table(index="date", columns="Quotation Source", values="price", aggfunc="first")
            .sort_index()
    )

    # (선택) 날짜가 모두 결측인 행 제거
    df_price_pivot = df_price_pivot.dropna(how="all")
    df.head()
    dfs.append(df_price_pivot)
df_price = pd.concat(dfs, ignore_index=False)

In [ ]:
# df_price 결측치 보정 by linear interpolation
df_price = df_price.interpolate(method='linear')


In [6]:
# 결측치 확인
print(df_price.isnull().sum())
print(df_demand.isnull().sum())
print(df_weather.isnull().sum()) # 날씨 결측치는 강수량이 없다는 의미 -> 처리 필요 없음


Quotation Source
PLATT'S AP/AG    0
PLATT'S CRUDE    0
dtype: int64
date      0
demand    0
dtype: int64
date              0
ta_avg_mean       0
rn_day_mean    1846
dtype: int64


# feature 생성
이제 아래에 위 raw data들 가지고 각자 feature 생성해주면 됩니다~ 
최종 dataframe은 df 로 모든 feature를 넣을 예정!

In [7]:
# season feature
df = df_demand
df['season'] = df['date'].dt.month % 12 // 3 # winter: 0, spring: 1, summer: 2, autumn: 3

In [24]:
# 월별 평균 가격 df (monthly_price_df)
monthly_price_df = (
    df_price.resample("M").mean()
             .reset_index()
)
monthly_price_df["date"] = monthly_price_df["date"].dt.strftime("%Y%m")
monthly_price_df.head()


/var/folders/_m/6spg1_4j7797c0k1z1nnk84w0000gq/T/ipykernel_48110/2380502797.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_price.resample("M").mean()


Quotation Source,date,PLATT'S AP/AG,PLATT'S CRUDE
0,200101,118.215909,22.850909
1,200102,130.668750,24.803000
2,200103,132.869318,23.449545
3,200104,136.556250,24.183000
4,200105,145.747283,25.663043


In [25]:
# 월별 가격 변동률 df (mom_pct_df)
price_cols = ["PLATT'S AP/AG", "PLATT'S CRUDE"]
monthly_price_df[price_cols] = monthly_price_df[price_cols].apply(pd.to_numeric)

mom_pct_df = monthly_price_df[price_cols].pct_change().shift(-1) * 100
mom_pct_df.index = monthly_price_df["date"]
mom_pct_df.head()

Quotation Source,PLATT'S AP/AG,PLATT'S CRUDE
date,,
200101,10.533981,8.542728
200102,1.684081,-5.456818
200103,2.774856,3.127799
200104,6.730584,6.120181
200105,-8.693062,-0.035054


In [ ]:
# monthly_weight_df

